In [23]:
from numpy import average
import pandas as pd
import numpy as np
from scipy.stats import gstd

In [24]:
# Laod in modeled concentrations
well_sims = pd.read_csv("../ies/peterson_tran.6.obs.csv", index_col=False)  # rows = ensemble members, columns = observation names

#change "base" to a number 
well_sims.iloc[-1, 0] = str(int(well_sims.iloc[-2, 0]) + 1)

well_sims['real_name'] = well_sims['real_name'].astype('float64')

In [25]:
# Load phi.actual.csv
phi_df = pd.read_csv("../ies/peterson_tran.phi.actual.csv", index_col=False)  # Or phi.composite.csv if you're using that

# Extract only the phi values for the final iteration (e.g., iteration 10)
# Get the row with the largest iteration number
last_iter = phi_df['iteration'].max()
phi_row = phi_df[phi_df['iteration'] == last_iter]

#Get Standard deviation of Phi
phi_mean = phi_df.loc[last_iter, 'mean']

# Get just the phi values for realizations (these are usually in columns labeled as ints)
phi_values = phi_row.iloc[:, 6:]
phi_values = phi_values.rename(columns={'base':str(len(phi_values.columns) - 1)})  # rename base
phi_values.columns = phi_values.columns.astype('float64')

# Match realizations from Phi and wells
realizations = well_sims['real_name']
phi_mask = np.isin(phi_values.columns, realizations)
phi_values = phi_values.loc[:, phi_mask]

# value reduction
phi_vals = phi_values / phi_mean

In [26]:
weights = np.exp(-0.5 * phi_vals)
weights /= weights.sum()

In [27]:
def weighted_percentile(data, weights, percentile):
    """Compute the weighted percentile."""
    sorter = np.argsort(data)
    data_sorted = data[sorter]
    weights_sorted = weights[sorter]
    cum_weights = np.cumsum(weights_sorted)
    normed_cum_weights = cum_weights / cum_weights[-1]
    return np.interp(percentile, normed_cum_weights, data_sorted)

In [28]:
weighted_stats = {}
for col in well_sims.columns[1:]:
    values = well_sims[col].values
    weighted_stats[col] = {
        "mean": np.average(values, weights=weights.values[0]),
        "standard_deviation":np.std(values),
        "geometric_standard_deviation": gstd(values),
        "standard_error":np.std(values) / len(values),
        "lower_95ci": weighted_percentile(values, weights.values[0], 0.025),
        "upper_95ci": weighted_percentile(values, weights.values[0], 0.975), 
    }

ci_weighted_df = pd.DataFrame(weighted_stats).T
print(ci_weighted_df)

                          mean  standard_deviation  \
brd_6_11138.0_pfos         NaN        1.400870e-05   
11960_opr_2_11138.0_pfos   NaN        6.632297e-12   
12425_opr_1_11138.0_pfos   NaN        6.381804e-12   
dle_11138.0_pfos           NaN        5.604232e-10   
gld_3_11138.0_pfos         NaN        1.012785e-13   
...                        ...                 ...   
w9_18077.0_pfhxs           NaN        3.342322e-03   
w12_18077.0_pfhxs          NaN        3.130814e-03   
c2_18077.0_pfhxs           NaN        1.351037e-04   
c36_18077.0_pfhxs          NaN        3.653289e-04   
w7_18077.0_pfhxs           NaN        1.740533e-04   

                          geometric_standard_deviation  standard_error  \
brd_6_11138.0_pfos                            1.739177    5.812738e-08   
11960_opr_2_11138.0_pfos                   2665.917878    2.751990e-14   
12425_opr_1_11138.0_pfos                   4973.450181    2.648051e-14   
dle_11138.0_pfos                             24.125818 